# Path B-lite: Weighted KDE @ n=50 (Option B)

**Date:** 2026-04-18.
**Hypothesis:** With a larger training pool (n=50 instead of n=20), weighted KDE has a better chance of finding 1-2 truly similar candidates for outlier targets. High-similarity training can dominate the shape even when most of the pool is dissimilar.

**Configs compared (all at T-3d, same combined_score α=0.5, σ_gap=8, ceil=0.7d):**
- `ship` — unweighted, n=20 (current ship stack)
- `unweighted_n50` — unweighted, n=50 (isolates pool-size effect)
- `weighted_n20` — weighted, n=20 (from previous experiment)
- `weighted_n50` — weighted, n=50 (the candidate)

This lets us see pool-size effect, weighting effect at each n, and combined effect.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name != 'notebooks':
    ROOT = ROOT / 'notebooks'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from _helpers import (
    reviews, close_date_map, gap_lookup, first_review_ts, gap_for_slug,
    combined_score_with_scores,
    snapshot_state, passes_skip_rules_for_snap,
    build_critic_profiles, build_kde_lambda_model_capped,
    build_weighted_critic_profiles, build_weighted_kde_lambda_model,
    predict_window_custom,
    CACHE_DIR,
)

SHIP_ALPHA = 0.5
SHIP_SIGMA_GAP = 8.0
SHIP_BANDWIDTH_FLOOR = 0.5
SHIP_BANDWIDTH_CEIL = 0.7
SNAP = 3.0

CACHE = CACHE_DIR / 'path_b_lite_weighted_n50.pkl'
print('Ready.')

## LOO across cohort — all 4 configs

In [ ]:
def run_config(target, target_gap, state, midnight_utc_dbc, n_training, weighted):
    target_window_days = state['first_review_dbc'] - SNAP
    target_critics = state['observed_critics']

    scores = combined_score_with_scores(
        target, target_gap, target_critics, target_window_days,
        k=n_training, alpha=SHIP_ALPHA, sigma_gap=SHIP_SIGMA_GAP,
    )
    if len(scores) < 5:
        return None

    if weighted:
        profiles = build_weighted_critic_profiles(reviews, close_date_map, scores, verbose=False)
        model = build_weighted_kde_lambda_model(
            profiles,
            bandwidth_floor=SHIP_BANDWIDTH_FLOOR,
            bandwidth_ceiling=SHIP_BANDWIDTH_CEIL,
        )
    else:
        profiles = build_critic_profiles(reviews, close_date_map, list(scores.keys()), verbose=False)
        model = build_kde_lambda_model_capped(
            profiles,
            bandwidth_floor=SHIP_BANDWIDTH_FLOOR,
            bandwidth_ceiling=SHIP_BANDWIDTH_CEIL,
        )

    pred = predict_window_custom(
        model, dbc_from=SNAP, dbc_to=midnight_utc_dbc,
        observed_critics=target_critics,
        observed_count=state['observed_count'],
        first_review_dbc=state['first_review_dbc'],
    )
    return float(pred)


def run_all(force=False):
    if CACHE.exists() and not force:
        return pd.read_pickle(CACHE)

    configs = [
        ('ship',           20, False),
        ('unweighted_n50', 50, False),
        ('weighted_n20',   20, True),
        ('weighted_n50',   50, True),
    ]

    rows = []
    for i, target in enumerate(close_date_map):
        target_gap = gap_for_slug(target)
        if target_gap is None:
            continue
        target_close = close_date_map[target]
        midnight_utc_dbc = (target_close - target_close.floor('D')).total_seconds() / 86400
        snap_time = target_close - pd.Timedelta(days=SNAP)
        state = snapshot_state(target, snap_time)
        passed, _ = passes_skip_rules_for_snap(state, SNAP)
        if not passed:
            continue

        mr = reviews[reviews['movie_slug'] == target].copy()
        mr['dbc'] = (target_close - mr['estimated_timestamp']).dt.total_seconds() / 86400
        actual_p1 = int(((mr['dbc'] > midnight_utc_dbc) & (mr['dbc'] <= SNAP)).sum())

        row = {
            'target': target,
            'target_gap': target_gap,
            'observed_count': state['observed_count'],
            'actual_phase1': actual_p1,
        }
        for name, n_tr, weighted in configs:
            try:
                pred = run_config(target, target_gap, state, midnight_utc_dbc, n_tr, weighted)
                row[f'pred_{name}'] = pred
                row[f'err_{name}'] = pred - actual_p1 if pred is not None else np.nan
                row[f'abs_err_{name}'] = abs(pred - actual_p1) if pred is not None else np.nan
            except Exception as e:
                print(f'{target} {name} failed: {e}')
                row[f'pred_{name}'] = np.nan
                row[f'err_{name}'] = np.nan
                row[f'abs_err_{name}'] = np.nan
        rows.append(row)

        if (i + 1) % 30 == 0:
            print(f'  {i+1}/{len(close_date_map)} targets')

    df = pd.DataFrame(rows)
    df.to_pickle(CACHE)
    print(f'Cached {len(df)} rows')
    return df

results = run_all()
print(f'\nn={len(results)}')

## Aggregate MAE by config, stratified by quartile

In [ ]:
CONFIGS = ['ship', 'unweighted_n50', 'weighted_n20', 'weighted_n50']

results['q_actual'] = pd.qcut(results['actual_phase1'], q=4, labels=['Q1','Q2','Q3','Q4'], duplicates='drop')

def cohort_table(df, label):
    print(f'\n{label} (n={len(df)}):')
    print(f'  {"config":18s}  {"MAE":>6s}  {"mean_err":>9s}  {"delta_vs_ship":>14s}')
    base_mae = df['abs_err_ship'].mean()
    for cfg in CONFIGS:
        mae = df[f'abs_err_{cfg}'].mean()
        me = df[f'err_{cfg}'].mean()
        if cfg == 'ship':
            delta_str = '     —'
        else:
            delta_pct = 100 * (base_mae - mae) / base_mae if base_mae > 0 else 0
            delta_str = f'{delta_pct:+13.1f}%'
        print(f'  {cfg:18s}  {mae:6.2f}  {me:+9.2f}  {delta_str}')

cohort_table(results, 'Full cohort')
for q in ['Q1','Q2','Q3','Q4']:
    sub = results[results['q_actual'] == q]
    if len(sub):
        lo, hi = int(sub['actual_phase1'].min()), int(sub['actual_phase1'].max())
        cohort_table(sub, f'{q} (actual [{lo}, {hi}])')

## H/m subset

In [ ]:
HM = ['the_drama', 'the_super_mario_galaxy_movie', 'forbidden_fruits_2026',
      'they_will_kill_you', 'you_me_and_tuscany']
hm = results[results['target'].isin(HM)].copy()
cols = ['target', 'actual_phase1'] + [f'pred_{c}' for c in CONFIGS]
print('H/m per-target predictions:')
print(hm[cols].to_string(index=False, float_format='%.2f'))
print()
print('H/m errors:')
err_cols = ['target', 'actual_phase1'] + [f'err_{c}' for c in CONFIGS]
print(hm[err_cols].to_string(index=False, float_format='%.2f'))
print()
cohort_table(hm, 'H/m aggregate (5 movies)')

## Decision

Key comparisons:
- **`unweighted_n50` vs `ship`**: isolates pool-size effect. Did just using 50 movies help?
- **`weighted_n50` vs `unweighted_n50`**: isolates the weighting effect at the larger pool.
- **`weighted_n50` vs `ship`**: combined effect (our candidate to ship).
- **h/m subset movement**: do the outliers actually find better neighbors at n=50?